# Lab 6 - Frequency Domain Filtering
### ARTI 404 – Image Processing

## Procedural Example: Laplacian Filter in Frequency Domain

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.transform import resize

# Load cameraman image (already grayscale)
image = data.camera()
image = resize(image, (256, 256))  # Resize for faster FFT (optional)

# Get image size
M, N = image.shape

# Generate frequency grid (centered)
u = np.fft.fftfreq(M).reshape(-1, 1)
v = np.fft.fftfreq(N).reshape(1, -1)

# Compute Laplacian filter in frequency domain
laplacian_filter = -4 * (np.pi ** 2) * (u**2 + v**2)

# Apply 2D FFT
F = np.fft.fft2(image)

# Apply Laplacian filter in frequency domain
F_lap = F * laplacian_filter

# Inverse FFT to get spatial result
laplacian_image = np.fft.ifft2(F_lap).real

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Cameraman')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(np.log(1 + np.abs(F_lap)), cmap='gray')
plt.title('Laplacian Spectrum')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(laplacian_image, cmap='gray')
plt.title('Laplacian (Spatial Output)')
plt.axis('off')

plt.tight_layout()
plt.show()

## Assessment Task #1: Sobel Filter in the Frequency Domain
Apply Sobel X and Sobel Y filters to the Cameraman image in the frequency domain and display the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.transform import resize
from numpy.fft import fft2, ifft2, ifftshift

# Load and resize image
image = data.camera()
image = resize(image, (256, 256))

# Sobel filters
sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float64)

sobel_y = np.array([[-1, -2, -1],
                     [ 0,  0,  0],
                     [ 1,  2,  1]], dtype=np.float64)

# Function: proper padding and centering of kernel in spatial domain
def center_embed_kernel(kernel, shape):
    padded = np.zeros(shape)
    kh, kw = kernel.shape
    ph, pw = shape
    # Find center coordinates
    cy, cx = ph // 2, pw // 2
    # Insert kernel centered at the image center
    padded[cy - kh//2:cy - kh//2 + kh, cx - kw//2:cx - kw//2 + kw] = kernel
    return fft2(ifftshift(padded))  # Shift before FFT to place (0,0) at top-left

# Compute FFT of image
F = fft2(image)

# FFT of centered Sobel filters (use the provided function)
H_x = center_embed_kernel(sobel_x, image.shape)
H_y = center_embed_kernel(sobel_y, image.shape)

# Multiply in the frequency domain
F_sobel_x = F * H_x
F_sobel_y = F * H_y

# Inverse FFT to get spatial results
sobel_x_result = np.abs(ifft2(F_sobel_x).real)
sobel_y_result = np.abs(ifft2(F_sobel_y).real)

# Combine Sobel X and Y to get the gradient magnitude
sobel_combined = np.sqrt(sobel_x_result**2 + sobel_y_result**2)

# Normalize for display
def normalize(img):
    return (img - img.min()) / (img.max() - img.min())

# Plot results
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: Original + Frequency Spectra
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Cameraman', fontsize=13)
axes[0, 0].axis('off')

axes[0, 1].imshow(np.log(1 + np.abs(F_sobel_x)), cmap='gray')
axes[0, 1].set_title('Sobel X - Frequency Spectrum', fontsize=13)
axes[0, 1].axis('off')

axes[0, 2].imshow(np.log(1 + np.abs(F_sobel_y)), cmap='gray')
axes[0, 2].set_title('Sobel Y - Frequency Spectrum', fontsize=13)
axes[0, 2].axis('off')

# Row 2: Spatial outputs
axes[1, 0].imshow(normalize(sobel_x_result), cmap='gray')
axes[1, 0].set_title('Sobel X - Spatial Output (Horizontal Edges)', fontsize=13)
axes[1, 0].axis('off')

axes[1, 1].imshow(normalize(sobel_y_result), cmap='gray')
axes[1, 1].set_title('Sobel Y - Spatial Output (Vertical Edges)', fontsize=13)
axes[1, 1].axis('off')

axes[1, 2].imshow(normalize(sobel_combined), cmap='gray')
axes[1, 2].set_title('Sobel Combined (Gradient Magnitude)', fontsize=13)
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()